# Art Genre Classifier

Fine-tunes an ImageNet ResNet152 to tell apart five painting genres from WikiArt: **abstract, cityscape,
portrait, landscape, animal painting** (2,000 images each).

This is a cleaned-up re-run of the 2023 notebook. What changed and why:

1. **Separate validation and test sets** (70/15/15). The old notebook early-stopped on the test set and
   reported the best test score, which overstates accuracy.
2. **Two-stage training.** First only the new head trains on a frozen ResNet, then the last 30 layers are
   unfrozen with a 100x smaller learning rate. Training both at once let the random head push large
   gradients into the pretrained layers.
3. **BatchNorm stays in inference mode** while fine-tuning, so small batches don't overwrite its statistics.
4. **Lighter regularisation** on the head (L2 1e-4 instead of 0.05).
5. **Random sample** of 2,000 images per genre instead of the first 2,000 rows.
6. **Seeded and deterministic**, and images stream from disk instead of being loaded into RAM.

In [ ]:
import json, os, random, glob
import numpy as np
import tensorflow as tf
import keras
from keras import layers, regularizers
from keras.applications import ResNet152
from keras.applications.resnet import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
keras.utils.set_random_seed(SEED)          # Python, NumPy and TensorFlow
tf.config.experimental.enable_op_determinism()

# Folder names in the dataset, in the model's output order (the API in backend/main.py uses the same order).
CLASSES = ["abstract", "cityscape", "portrait", "landscape", "animal-painting"]
PER_CLASS, SIZE, BATCH = 2000, 300, 32
OUT = "/kaggle/working"
print(tf.__version__, keras.__version__, tf.config.list_physical_devices("GPU"))

## Data

In [ ]:
# The dataset mount path differs between Kaggle versions, so find the genre folders.
root = next(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/abstract", recursive=True) if os.path.isdir(p))
print("images in", root)

rng = random.Random(SEED)
paths, labels = [], []
for label, name in enumerate(CLASSES):
    files = sorted(f for f in os.listdir(os.path.join(root, name)) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    rng.shuffle(files)
    chosen = files[:PER_CLASS]
    paths += [os.path.join(root, name, f) for f in chosen]
    labels += [label] * len(chosen)
    print(f"{name:16s} {len(files):6d} available, {len(chosen)} used")
paths, labels = np.array(paths), np.array(labels)

# 70% train, 15% validation (early stopping), 15% test (reported once at the end). Stratified by genre.
p_train, p_rest, y_train, y_rest = train_test_split(paths, labels, test_size=0.3, random_state=SEED, stratify=labels)
p_val, p_test, y_val, y_test = train_test_split(p_rest, y_rest, test_size=0.5, random_state=SEED, stratify=y_rest)
print(len(p_train), len(p_val), len(p_test))

In [ ]:
def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (SIZE, SIZE))
    return preprocess_input(img), label

def make_ds(p, y, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if shuffle:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).ignore_errors().batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(p_train, y_train, True), make_ds(p_val, y_val, False), make_ds(p_test, y_test, False)

## Model

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(20 / 360, seed=SEED),
    layers.RandomTranslation(0.15, 0.15, seed=SEED),
    layers.RandomZoom(0.15, seed=SEED),
], name="augment")

base = ResNet152(weights="imagenet", include_top=False, input_shape=(SIZE, SIZE, 3))
base.trainable = False

inputs = keras.Input((SIZE, SIZE, 3))
x = augment(inputs)
x = base(x, training=False)   # BatchNorm stays in inference mode, also when fine-tuning
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(1024, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
model = keras.Model(inputs, outputs)

### Stage 1: train the head on a frozen ResNet

In [ ]:
model.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", ["accuracy"])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=[
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)])

### Stage 2: fine-tune the last 30 layers (BatchNorm layers stay frozen)

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers[-30:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", ["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])

## Results on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
y_pred = np.argmax(model.predict(test_ds), axis=1)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
print(f"Test accuracy: {test_acc:.4f}")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
print(cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for i, (m, title) in enumerate([("accuracy", "Accuracy"), ("loss", "Loss")]):
    ax[i].plot(hist[m], label="train"); ax[i].plot(hist["val_" + m], label="validation")
    ax[i].axvline(len(h1.history["loss"]) - 0.5, color="grey", ls="--", lw=1)
    ax[i].set_title(title + " (dashed line: start of fine-tuning)"); ax[i].set_xlabel("epoch"); ax[i].legend()
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax[2], xticks_rotation=45, colorbar=False)
ax[2].set_title(f"Test set, accuracy {test_acc:.1%}")
plt.tight_layout(); plt.savefig(f"{OUT}/training.png", dpi=120); plt.show()

## Save

In [ ]:
model.save(f"{OUT}/art_genre_classifier.keras")
json.dump({
    "classes": CLASSES, "seed": SEED, "split": {"train": len(p_train), "val": len(p_val), "test": len(p_test)},
    "test_accuracy": float(test_acc), "test_loss": float(test_loss),
    "history": {k: [float(v) for v in vs] for k, vs in hist.items()},
    "stage1_epochs": len(h1.history["loss"]),
    "confusion_matrix": cm.tolist(),
}, open(f"{OUT}/results.json", "w"), indent=2)
print(os.listdir(OUT))